# Аксиора: подготовка данных для OCR

**Этап 2, ноутбук 1 из 2.** Можно использовать обычную CPU-среду Colab: GPU понадобится во втором ноутбуке.
Запускайте ячейки по порядку. Результат — ZIP с изображениями, LaTeX-разметкой и отдельными
train/validation/test. По умолчанию архив сохраняется на ваш Google Drive.

Источник реального почерка — [Google MathWriting](https://github.com/google-research/google-research/tree/master/mathwriting).
Его данные имеют лицензию **CC BY-NC-SA 4.0**: стандартный сценарий предназначен для некоммерческих исследований.
Полный исходный архив занимает около 2,9 ГБ и загружается во временный диск Colab; в подготовленную
выборку попадёт только заданное число формул. Для быстрой проверки переключите `DATASET_VARIANT` на `excerpt`.
Excerpt содержит лишь по 100 реальных формул на исходную выборку и не подходит для качественного обучения.

Если Google AI Pro подключён к другому аккаунту, откройте Colab и подключите Drive под этим аккаунтом.
Google AI-планы включают преимущества Colab; доступный баланс и ускорители проверьте в панели ресурсов.
Настройки ниже работают и в базовом сценарии без подписки.

Синтетические печатные примеры — небольшое дополнение, а не имитация настоящего почерка.
Модель обучается одной вырезанной формуле; распознавание целой страницы и проверка решения — следующие этапы.


In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/danilsamchenckodmitrievich-netizen/Laba-proektx.git"
SOURCE_REF = "main"  # Можно указать ветку; фактический commit попадёт в отчёт.
REPO_DIR = Path("/content/axiora")
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", SOURCE_REF, REPO_URL, str(REPO_DIR)], check=True)
else:
    actual_origin = subprocess.check_output(["git", "-C", str(REPO_DIR), "remote", "get-url", "origin"], text=True).strip()
    if actual_origin != REPO_URL:
        raise RuntimeError("В /content/axiora находится другой проект. Выберите новый REPO_DIR.")
SOURCE_COMMIT = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
sys.path.insert(0, str(REPO_DIR))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "training/requirements-colab.txt")], check=True)
print("Версия исходников:", SOURCE_COMMIT)


## Настройки данных

`full` — реальное обучение, `excerpt` — быстрая проверка, `none` — только собственные изображения.
Для собственного набора укажите путь к JSONL: каждая строка содержит `image`, `latex`, `group_id`
и при необходимости `split` (`train`, `validation`, `test`). Изображения лежат рядом с JSONL или в его подпапках.
Один `group_id` обозначает автора/сессию и всегда остаётся в одной выборке. Не используйте имена учеников как идентификаторы.
Подробный пример формата — `training/README.md` в репозитории. Укажите условия использования собственных данных.


In [ ]:
DATASET_VARIANT = "full"
LIMITS = {"train": 12000, "validation": 1000, "test": 500}
SYNTHETIC_COUNT = 300  # Дополнительно ограничивается 10% от основного train.
CUSTOM_MANIFEST = ""  # Например: /content/my-formulas/labels.jsonl
CUSTOM_LICENSE = "unspecified"
SEED = 42
USE_GOOGLE_DRIVE = True
DATASET_DIR = Path("/content/axiora-ocr-data")


In [ ]:
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    PERSIST_DIR = Path("/content/drive/MyDrive/Axiora/ocr")
else:
    PERSIST_DIR = Path("/content/axiora-artifacts")
PERSIST_DIR.mkdir(parents=True, exist_ok=True)
print("Артефакты и контрольные точки:", PERSIST_DIR)


## Загрузка источника

Во время загрузки полного архива оставьте вкладку Colab открытой. GPU для этой операции не нужен.
Малый excerpt дополнительно проверяется по закреплённой SHA-256; для полного архива фактический хэш сохранится в отчёте.


In [ ]:
from training.data import download_mathwriting
import time

archive_path = None
if DATASET_VARIANT != "none":
    last_message = [0.0]
    def progress(written, total):
        if time.monotonic() - last_message[0] > 5:
            print(f"Загружено {written / 2**20:.0f} МБ" + (f" из {total / 2**20:.0f}" if total else ""))
            last_message[0] = time.monotonic()
    archive_path = download_mathwriting(DATASET_VARIANT, Path("/content") / f"mathwriting-{DATASET_VARIANT}.tgz", progress)
    print("Архив готов:", archive_path.name)


## Растеризация и формирование выборок

Официальные разбиения MathWriting сохраняются. Исходные writer ID не опубликованы; нельзя утверждать,
что все авторы во всех выборках различаются. Для собственных данных применяется разделение по `group_id`.
Идентичные изображения удаляются с приоритетом test → validation → train. Печатные примеры попадают только в train.


In [ ]:
from training.data import prepare_dataset, audit_dataset
import json

if not DATASET_DIR.exists():
    prepare_dataset(
        archive_path, DATASET_DIR, LIMITS, variant=DATASET_VARIANT,
        synthetic_count=SYNTHETIC_COUNT, custom_manifest=CUSTOM_MANIFEST or None,
        custom_license=CUSTOM_LICENSE, seed=SEED,
    )
else:
    print("Набор уже существует: проверяем его. Для другой конфигурации задайте новый DATASET_DIR.")
dataset_audit = audit_dataset(DATASET_DIR)
print(json.dumps(dataset_audit, ensure_ascii=False, indent=2))


## Проверьте изображения и подписи

Просмотрите несколько примеров перед обучением: формула должна целиком помещаться в кадр, а LaTeX соответствовать почерку.
Это просмотр данных, а не оценка качества OCR.


In [ ]:
from training.data import load_manifest
from PIL import Image
import matplotlib.pyplot as plt
import random

rows = load_manifest(DATASET_DIR, "train")
selected = random.Random(SEED).sample(rows, min(9, len(rows)))
figure, axes = plt.subplots(3, 3, figsize=(12, 9))
for axis, row in zip(axes.flat, selected):
    with Image.open(DATASET_DIR / row["image"]) as image:
        axis.imshow(image)
    axis.set_title(row["latex"][:65], fontsize=9)
for axis in axes.flat:
    axis.axis("off")
plt.tight_layout()
plt.show()


## Сохранение для второго ноутбука

Один ZIP переносится между сессиями быстрее, чем тысячи мелких файлов на Drive.
Исходные картинки и разметка не загружаются на Hugging Face автоматически.


In [ ]:
from training.data import zip_dataset

DATASET_ZIP = PERSIST_DIR / "axiora-ocr-dataset.zip"
checksum = zip_dataset(DATASET_DIR, DATASET_ZIP)
DATASET_ZIP.with_suffix(".zip.sha256").write_text(checksum + "\n")
print("Готово:", DATASET_ZIP)
print("SHA-256:", checksum)
if not USE_GOOGLE_DRIVE:
    from google.colab import files
    files.download(str(DATASET_ZIP))
    files.download(str(DATASET_ZIP.with_suffix(".zip.sha256")))


Далее откройте `02_train_ocr_colab.ipynb`, выберите **Среда выполнения → Сменить среду выполнения → T4 GPU**.
С Drive набор подхватится по стандартному пути. Для excerpt во втором ноутбуке нужен `RUN_MODE = "smoke"`.
